# Silverwing-ML: GPU Training

1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Run cells in order
3. Checkpoints save to Google Drive

In [ ]:
# Cell 1: Mount Google Drive (checkpoints save here)
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/silverwing'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints save to: {DRIVE_DIR}')

In [ ]:
# Cell 2: Clone project from GitHub
!git clone https://github.com/oledesug-source/silverwing-ml.git /content/Silverwing-ML
os.chdir('/content/Silverwing-ML')
print(f'Project at: {os.getcwd()}')

In [ ]:
# Cell 3: Check GPU + install deps
!nvidia-smi
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q pyyaml numpy
import torch
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# Cell 4: Run tests
!python -m pytest tests/ -q --tb=short

In [ ]:
# Cell 5: Pretrain 5000 steps (saves to Drive)
!python scripts/train.py --config configs/training.yaml --device cuda --max-steps 5000 --batch-size 4 --checkpoint-dir $DRIVE_DIR/pretrain --no-clean-repo-check

In [ ]:
# Cell 6: SFT (saves to Drive)
!python scripts/train_sft.py --config configs/sft_combined.yaml --init-from $DRIVE_DIR/pretrain/best.pt --device cuda --checkpoint-dir $DRIVE_DIR/sft-combined --no-clean-repo-check

In [ ]:
# Cell 7: List checkpoints on Drive
import os
for root, dirs, files in os.walk(DRIVE_DIR):
    for f in sorted(files):
        if f.endswith('.pt'):
            size_mb = os.path.getsize(os.path.join(root, f)) / 1e6
            print(f'{f} ({size_mb:.0f} MB)')